# Figure 1c,d,e - composite panels

Regenerates panels **c**, **d** and **e** of Figure 1 as *standalone* PNGs, to be
tiled with the panel a schematic and the panel b band diagram by hand
(PowerPoint).

- **(c)** bulk CrSBr SQUID magnetometry at 10 K, $M/M_\mathrm{S}$ along all three crystal axes: sharp spin-flip at 0.3 T along $b$, canting to saturation at 1 T along $a$ and 2 T along $c$;
- **(d)** junction current versus field along the hard $c$-axis, $H_\mathrm{z}$, saturating near 2 T as the magnetization cants;
- **(e)** junction current versus field along the easy $b$-axis, $H_\mathrm{y}$, with the layer-by-layer spin-flip transition at 0.3 T.

Panels d and e are the 20 K field sweeps with the junction current read off each
Gaussian-filtered $I(V)$ curve at the probe bias $V = +0.5\,$V, as the
manuscript caption states. The trace is kept in **acquisition order**, so the
sweep direction and the hysteresis of the $b$-axis loop survive as a connected
line.

All three panels are drawn at the same figure size and the project RC sizes
(labels 22, ticks 18), so they scale together.

**Colours follow the Figure 1 caption**, which is the source of truth for this
figure: $a$-axis orange, $b$-axis blue, $c$-axis bluish green. Panel d is a
$c$-axis sweep and so carries the same green as the $c$-axis curve in panel c;
panel e the same blue.

Data:
- panel c: `data/SQUID_bulk_CrSBr/MvH/*.rso.dat` (MPMS RSO sweeps at 10 K)
- panels d,e: `output/IV_H_scans/dataframes/{b,c}_scans/IV_gaussian_20K.pkl`

Loading and normalisation live in `scripts/make_fig1_panels.py`; this notebook
only orchestrates and plots.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Resolve project root (works whether cwd is the repo root or notebooks/...).
ROOT = Path.cwd().resolve()
while not (ROOT / "scripts").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.make_fig1_panels import (  # noqa: E402
    SQUID_TRACES,
    load_panel_trace,
    load_squid_trace,
)
from scripts.utils.notebook_setup import OKABE_ITO, configure_plot_style  # noqa: E402

configure_plot_style()

In [ ]:
# ===================== Parameters =====================
V_PROBE = 0.5        # V, probe bias at which the junction current is read
TEMPERATURE = "20K"  # dataframe temperature key for panels d, e
FIGSIZE = (6, 5)     # identical for all three panels, so they scale together
DPI = 300

# Figure 1 caption colours (source of truth for this figure).
COLOR_A_AXIS = OKABE_ITO["orange"]
COLOR_B_AXIS = OKABE_ITO["blue"]
COLOR_C_AXIS = OKABE_ITO["bluish_green"]

OUTDIR = ROOT / "output" / "paper figures" / "fig1_panels"
OUTDIR.mkdir(parents=True, exist_ok=True)

print(f"writing to {OUTDIR}")

In [ ]:
# ---------------- Panel c: bulk SQUID M(H), three axes, 10 K ----------------
traces = [load_squid_trace(fname, label) for label, fname, _ in SQUID_TRACES]
colors = [COLOR_A_AXIS, COLOR_B_AXIS, COLOR_C_AXIS]

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
for trace, color in zip(traces, colors):
    ax.errorbar(
        trace.field_T,
        trace.M_norm,
        trace.err_norm,
        fmt="-o",
        markersize=3,
        label=trace.label,
        color=color,
    )
ax.set_xlabel(r"$H$ (T)")
ax.set_ylabel(r"$M/M_\mathrm{S}$")
ax.set_xticks(np.arange(-3, 4, 1))
ax.legend(loc="best", frameon=False)

outfile_c = OUTDIR / "fig1c_squid_MvsH_10K.png"
fig.savefig(outfile_c, dpi=DPI)
plt.show()

print(f"panel c -> {outfile_c.name}")
for trace in traces:
    print(
        f"  {trace.label}: {len(trace.field_T)} points, "
        f"H = {trace.field_T.min():+.2f} to {trace.field_T.max():+.2f} T, "
        f"M_S = {trace.sat_moment:.3e} A m^2"
    )

In [ ]:
# ---------------- Panel d: hard c-axis, H_z ----------------
field, current_nA, v_actual = load_panel_trace("c_scans", V_PROBE, TEMPERATURE)

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
ax.plot(field, current_nA, "o-", markersize=4, linewidth=1.5, color=COLOR_C_AXIS)
ax.set_xlabel(r"$H_\mathrm{Z}$ (T)")
ax.set_ylabel(r"$I$ (nA)")

outfile_d = OUTDIR / "fig1d_c_axis_20K.png"
fig.savefig(outfile_d, dpi=DPI)
plt.show()

print(f"panel d -> {outfile_d.name}")
print(f"  field steps : {len(field)}")
print(f"  H range     : {field.min():+.3f} to {field.max():+.3f} T")
print(f"  I range     : {current_nA.min():.2f} to {current_nA.max():.2f} nA")
print(f"  probe bias  : {v_actual:+.4f} V (nearest grid point to {V_PROBE:+.2f} V)")

In [ ]:
# ---------------- Panel e: easy b-axis, H_y ----------------
field, current_nA, v_actual = load_panel_trace("b_scans", V_PROBE, TEMPERATURE)

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
ax.plot(field, current_nA, "o-", markersize=4, linewidth=1.5, color=COLOR_B_AXIS)
ax.set_xlabel(r"$H_\mathrm{Y}$ (T)")
ax.set_ylabel(r"$I$ (nA)")

outfile_e = OUTDIR / "fig1e_b_axis_20K.png"
fig.savefig(outfile_e, dpi=DPI)
plt.show()

print(f"panel e -> {outfile_e.name}")
print(f"  field steps : {len(field)}")
print(f"  H range     : {field.min():+.3f} to {field.max():+.3f} T")
print(f"  I range     : {current_nA.min():.2f} to {current_nA.max():.2f} nA")
print(f"  probe bias  : {v_actual:+.4f} V (nearest grid point to {V_PROBE:+.2f} V)")

## Assembling in PowerPoint

All three PNGs are 300 dpi at the same figure size, so drop them in at the same
scale and the fonts will match. Still to add by hand, as in the current
Figure 1: the red/blue spin-configuration arrow pairs in d and e (annotations,
not data) and the panel letters.